# 面试题：执行中如何动态重规划？

回答要点：外部事实变化时先 checkpoint 已验证动作与权威状态，保留未受影响的完成节点，仅替换受影响后缀。重规划要保留同一目标、预算、权限和幂等边界，记录触发事件与计划 diff。暂态读失败可重试，库存、预算和权限变化才应重新求解。

## 真实案例

采购助手已完成预算查询和型号选择，在提交订单前发现型号 M1 缺货；六个节点覆盖读取、替换、审批和提交。

## 基线

基线继续执行旧计划，会把缺货型号提交给采购系统。

## 结果解读

手写重规划器输出 checkpoint、受影响节点和替代子图。

## 失败案例

从头重跑会把已完成的预算查询误算为新的收费调用。

In [1]:
plan = [{'id':'R1','name':'查预算','deps':[]}, {'id':'R2','name':'选 M1 型号','deps':['R1']}, {'id':'R3','name':'创建 M1 草稿','deps':['R2']}, {'id':'R4','name':'经理审批','deps':['R3']}, {'id':'R5','name':'提交采购单','deps':['R3','R4']}, {'id':'R6','name':'通知员工','deps':['R5']}]  # 构造六个原计划节点及其依赖关系。
completed = {'R1','R2'}  # 记录已经由工具结果验证过的 checkpoint 节点。
event = {'type':'库存变化','model':'M1','available':False,'replacement':'M2'}  # 构造触发重规划的权威库存事件。
print('原计划:', [(node['id'], node['name']) for node in plan], '，已完成:', completed)  # 输出计划与稳定 checkpoint。

原计划: [('R1', '查预算'), ('R2', '选 M1 型号'), ('R3', '创建 M1 草稿'), ('R4', '经理审批'), ('R5', '提交采购单'), ('R6', '通知员工')] ，已完成: {'R1', 'R2'}


In [2]:
unsafe_next = [node['id'] for node in plan if node['id'] not in completed]  # 构造忽略库存变化的旧计划剩余步骤。
print('旧计划继续执行:', unsafe_next)  # 输出会继续创建 M1 草稿并提交的错误后缀。
print('基线风险：模型知道库存变化却没有把它变成确定性计划失效条件。')  # 解释文本认知不能替代状态变更。

旧计划继续执行: ['R3', 'R4', 'R5', 'R6']
基线风险：模型知道库存变化却没有把它变成确定性计划失效条件。


In [3]:
def replan(old_plan, done, trigger):  # 定义保留 checkpoint、替换受影响后缀的手写重规划器。
    affected = {'R2','R3','R4','R5','R6'} if trigger['type'] == '库存变化' else set()  # 根据事实变化标记依赖 M1 的节点。
    preserved = [node for node in old_plan if node['id'] in done and node['id'] not in affected]  # 保留已完成且未被事实推翻的节点。
    replacement = [{'id':'R2b','name':'选 ' + trigger['replacement'] + ' 型号','deps':['R1']}, {'id':'R3b','name':'创建替代草稿','deps':['R2b']}, {'id':'R4b','name':'经理审批替代','deps':['R3b']}, {'id':'R5b','name':'提交替代采购单','deps':['R3b','R4b']}, {'id':'R6b','name':'通知员工','deps':['R5b']}]  # 生成只覆盖受影响后缀的替代子图。
    return preserved, affected, replacement  # 返回 checkpoint、失效节点和新版后缀。

In [4]:
preserved, affected, replacement = replan(plan, completed, event)  # 对库存事件执行局部重规划。
print('保留 checkpoint:', [(node['id'], node['name']) for node in preserved])  # 输出不会重复运行的已验证步骤。
print('失效旧节点:', sorted(affected))  # 输出因库存事实而作废的计划后缀。
print('新后缀:', [(node['id'], node['name'], node['deps']) for node in replacement])  # 输出带依赖的替代型号计划。
print('重规划保留预算查询，只替换 M1 相关节点，因此不会重复副作用或浪费读调用。')  # 解读局部替换效果。

保留 checkpoint: [('R1', '查预算')]
失效旧节点: ['R2', 'R3', 'R4', 'R5', 'R6']
新后缀: [('R2b', '选 M2 型号', ['R1']), ('R3b', '创建替代草稿', ['R2b']), ('R4b', '经理审批替代', ['R3b']), ('R5b', '提交替代采购单', ['R3b', 'R4b']), ('R6b', '通知员工', ['R5b'])]
重规划保留预算查询，只替换 M1 相关节点，因此不会重复副作用或浪费读调用。


In [5]:
restart_cost = len(plan)  # 模拟从头重跑需要重新发出的六个动作数。
local_cost = len(replacement)  # 计算只重跑受影响后缀的动作数。
print('失败案例：从头重跑动作=', restart_cost, '，局部重规划动作=', local_cost)  # 展示 checkpoint 对成本与重复副作用的保护。
print('生产差距：需要持久化 plan version、工具 invocation id、预算快照、最大重规划次数与变更后的审批重新绑定。')  # 说明真实 replan 控制面。

失败案例：从头重跑动作= 6 ，局部重规划动作= 5
生产差距：需要持久化 plan version、工具 invocation id、预算快照、最大重规划次数与变更后的审批重新绑定。


In [6]:
assert [node['id'] for node in preserved] == ['R1']  # 验证只有未被库存事实推翻的预算查询被保留。
assert all('M2' not in node['name'] or node['id'] == 'R2b' for node in replacement)  # 验证替代型号出现在新的选择节点。
assert local_cost < restart_cost  # 验证局部重规划比全图重跑更少动作。